# Predicting Customer Ride Cancellations
## A Machine Learning Approach to Rideshare Marketplace Behavior

**Research question:** Can we predict whether a customer will cancel a ride using trip, pricing, waiting-time, location, and customer/driver characteristics?

This notebook compares **Cancelled by Customer (1)** with **Completed (0)** bookings. It excludes other outcomes from the primary analysis. Results therefore apply to these two groups, not to all ride requests.

**Important:** do not use variables that reveal the outcome after it happened. Before using a predictor, verify that it was available at the intended prediction time.

## Step 1 — Load and inspect the dataset

This notebook expects the uploaded CSV in `/mnt/data/ncr_ride_bookings(2).csv`. If you run the notebook locally, change `DATA_PATH` to the location of your CSV file.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

DATA_PATH = "/mnt/data/ncr_ride_bookings(2).csv"
df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())
display(df.head())
display(df.dtypes.to_frame("data_type"))
display(df.isna().sum().sort_values(ascending=False).to_frame("missing_count"))
display(df["Booking Status"].value_counts(dropna=False).to_frame("count"))

## Step 2 — Define the analytic sample and outcome

Primary comparison: customer-cancelled bookings versus completed bookings. We do not label driver cancellations, no-driver bookings, or incomplete rides as ordinary non-cancellations.

In [ ]:
# Clean whitespace and accidental surrounding quotation marks in text columns
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].str.strip().str.strip('"')

print(df["Booking Status"].value_counts(dropna=False))

analysis_df = df[
    df["Booking Status"].isin(["Cancelled by Customer", "Completed"])
].copy()

analysis_df["customer_cancelled"] = (
    analysis_df["Booking Status"] == "Cancelled by Customer"
).astype(int)

print("Analytic sample size:", len(analysis_df))
display(analysis_df["customer_cancelled"].value_counts().to_frame("count"))
display((analysis_df["customer_cancelled"].value_counts(normalize=True)
         .mul(100).round(2).to_frame("percent")))
display(pd.crosstab(analysis_df["Booking Status"],
                    analysis_df["customer_cancelled"]))

## Step 3 — Audit target leakage and predictor availability

The fields below should not be model predictors because they directly reveal the outcome or are usually observed after the ride. IDs are also excluded as direct predictors.

`Avg VTAT`, `Avg CTAT`, and `Booking Value` require a timing check. In this first conservative model, they are excluded until you verify that they were available when the prediction would be made.

In [ ]:
leakage_or_exclusion_cols = [
    "Booking Status",
    "Cancelled Rides by Customer",
    "Reason for cancelling by Customer",
    "Cancelled Rides by Driver",
    "Driver Cancellation Reason",
    "Incomplete Rides",
    "Incomplete Rides Reason",
    "Customer Rating",
    "Driver Ratings",
    "Booking ID",
    "Customer ID"
]

candidate_predictors = [
    "Vehicle Type", "Pickup Location", "Drop Location",
    "Ride Distance", "Booking Value", "Avg VTAT", "Avg CTAT",
    "Payment Method", "Date", "Time"
]

print("Exclusion columns present:",
      [c for c in leakage_or_exclusion_cols if c in analysis_df.columns])
print("Candidate columns present:",
      [c for c in candidate_predictors if c in analysis_df.columns])

for col in ["Avg VTAT", "Avg CTAT", "Booking Value", "Ride Distance", "Payment Method"]:
    if col in analysis_df.columns:
        print(f"\n{col} by outcome:")
        display(analysis_df.groupby("Booking Status", dropna=False)[col]
                .agg(["count", "mean", "median"]))

## Step 4 — Clean dates and engineer time features

This creates hour, day of week, month, weekend, peak-hour, and distance-category features. Peak hours are an example definition and should be checked in sensitivity analyses.

In [ ]:
analysis_df["Date"] = pd.to_datetime(analysis_df["Date"], errors="coerce")
analysis_df["Time_parsed"] = pd.to_timedelta(analysis_df["Time"], errors="coerce")
analysis_df["booking_datetime"] = (
    analysis_df["Date"] + analysis_df["Time_parsed"]
)

analysis_df["hour"] = analysis_df["booking_datetime"].dt.hour
analysis_df["day_of_week"] = analysis_df["booking_datetime"].dt.dayofweek
analysis_df["month"] = analysis_df["booking_datetime"].dt.month
analysis_df["is_weekend"] = (analysis_df["day_of_week"] >= 5).astype("Int64")
analysis_df["is_peak"] = (
    analysis_df["hour"].between(7, 9) |
    analysis_df["hour"].between(17, 19)
).astype("Int64")

analysis_df["distance_category"] = pd.cut(
    analysis_df["Ride Distance"],
    bins=[-np.inf, 3, 8, 15, np.inf],
    labels=["0-3", "3-8", "8-15", "15+"]
)

display(analysis_df[[
    "booking_datetime", "hour", "day_of_week", "month",
    "is_weekend", "is_peak", "Ride Distance", "distance_category"
]].head())
print("Rows with unparseable booking datetime:",
      analysis_df["booking_datetime"].isna().sum())

## Step 5 — Exploratory data analysis (EDA)

Describe patterns first; do not interpret group differences as causal effects.

In [ ]:
print(f"Customer cancellation rate in the analytic sample: "
      f"{analysis_df['customer_cancelled'].mean():.2%}")

hourly_rate = analysis_df.groupby("hour", dropna=False)["customer_cancelled"].mean() * 100
hourly_rate.plot(kind="bar", figsize=(10, 4))
plt.title("Customer Cancellation Rate by Hour")
plt.xlabel("Hour of day")
plt.ylabel("Cancellation rate (%)")
plt.tight_layout()
plt.show()

weekday_rate = analysis_df.groupby("day_of_week", dropna=False)["customer_cancelled"].mean() * 100
weekday_rate.plot(kind="bar", figsize=(8, 4))
plt.title("Customer Cancellation Rate by Day of Week")
plt.xlabel("Day of week (Monday = 0)")
plt.ylabel("Cancellation rate (%)")
plt.tight_layout()
plt.show()

for col in ["Ride Distance", "Avg VTAT", "Booking Value"]:
    if col in analysis_df.columns:
        print(f"\n{col} by outcome:")
        display(analysis_df.groupby("Booking Status")[col]
                .agg(["count", "mean", "median", "std"]).round(2))

for col in ["Vehicle Type", "Payment Method"]:
    if col in analysis_df.columns:
        print(f"\nCancellation rate by {col}:")
        display(analysis_df.groupby(col, dropna=False)["customer_cancelled"]
                .agg(bookings="size", cancellation_rate="mean")
                .sort_values("cancellation_rate", ascending=False)
                .round(3))

## Step 6 — Build the modeling dataset

The conservative first model uses booking time, distance, vehicle type, pickup/drop-off location, payment method, and distance category. It excludes ratings, cancellation-reason fields, IDs, and timing-ambiguous variables.

In [ ]:
numeric_features = [
    "hour", "day_of_week", "month", "is_weekend", "is_peak", "Ride Distance"
]
categorical_features = [
    "Vehicle Type", "Pickup Location", "Drop Location",
    "Payment Method", "distance_category"
]
feature_cols = [
    c for c in numeric_features + categorical_features
    if c in analysis_df.columns
]

model_df = analysis_df.dropna(subset=["booking_datetime"]).copy()
model_df = model_df.sort_values("booking_datetime").reset_index(drop=True)

X = model_df[feature_cols].copy()
y = model_df["customer_cancelled"].copy()

print("Features:", feature_cols)
print("X shape:", X.shape, "| y shape:", y.shape)
display(X.head())

## Step 7 — Chronological train/test split

Train on earlier bookings and test on later bookings to better approximate predicting future bookings. Check the date ranges and outcome prevalence in each split.

In [ ]:
split_idx = int(len(model_df) * 0.80)

X_train = X.iloc[:split_idx].copy()
X_test = X.iloc[split_idx:].copy()
y_train = y.iloc[:split_idx].copy()
y_test = y.iloc[split_idx:].copy()

print("Train period:",
      model_df["booking_datetime"].iloc[0], "to",
      model_df["booking_datetime"].iloc[split_idx - 1])
print("Test period:",
      model_df["booking_datetime"].iloc[split_idx], "to",
      model_df["booking_datetime"].iloc[-1])
print("Train size:", len(X_train), "| Test size:", len(X_test))
print("Train cancellation rate:", round(y_train.mean(), 3))
print("Test cancellation rate:", round(y_test.mean(), 3))

## Step 8 — Logistic Regression baseline

Preprocessing is fitted only on training data through a pipeline. Class weighting can help the model pay attention to cancellations, but predicted probabilities may need calibration before being interpreted as real-world risk.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

num_cols = [c for c in numeric_features if c in X_train.columns]
cat_cols = [c for c in categorical_features if c in X_train.columns]

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, num_cols),
    ("cat", categorical_transformer, cat_cols)
])

logistic_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(
        max_iter=2000, class_weight="balanced", random_state=42
    ))
])

logistic_model.fit(X_train, y_train)
print("Logistic Regression fitted.")

### Interpret Logistic Regression coefficients

For numeric variables, standardization means the odds ratio corresponds to a one-standard-deviation increase. Categorical coefficients are relative to the encoder's reference category. These are associations, not causal effects.

In [ ]:
feature_names_lr = logistic_model.named_steps["preprocessor"].get_feature_names_out()
coefficients = logistic_model.named_steps["model"].coef_[0]

coef_df = pd.DataFrame({
    "feature": feature_names_lr,
    "coefficient": coefficients,
    "odds_ratio": np.exp(np.clip(coefficients, -20, 20))
})

display(coef_df.sort_values("coefficient", ascending=False).head(15))
display(coef_df.sort_values("coefficient", ascending=True).head(15))

## Step 9 — XGBoost model

This model can capture nonlinear patterns and interactions. This is a starting configuration, not a tuned final model.

In [ ]:
from xgboost import XGBClassifier

negative_count = int((y_train == 0).sum())
positive_count = int((y_train == 1).sum())
scale_pos_weight = negative_count / positive_count

xgb_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", XGBClassifier(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="binary:logistic",
        eval_metric="logloss",
        scale_pos_weight=scale_pos_weight,
        random_state=42,
        n_jobs=-1
    ))
])

xgb_model.fit(X_train, y_train)
print("XGBoost fitted.")

## Step 10 — Evaluate and compare models

Report ROC-AUC, average precision (PR-AUC-style summary), precision, recall, F1, and accuracy. The default threshold of 0.5 is only a starting point.

In [ ]:
from sklearn.metrics import (
    roc_auc_score, average_precision_score, precision_score,
    recall_score, f1_score, accuracy_score, confusion_matrix,
    classification_report, precision_recall_curve
)
from sklearn.dummy import DummyClassifier

def evaluate_model(model, X_eval, y_eval, threshold=0.5):
    probabilities = model.predict_proba(X_eval)[:, 1]
    predictions = (probabilities >= threshold).astype(int)
    metrics = {
        "ROC_AUC": roc_auc_score(y_eval, probabilities),
        "Average_Precision": average_precision_score(y_eval, probabilities),
        "Precision": precision_score(y_eval, predictions, zero_division=0),
        "Recall": recall_score(y_eval, predictions, zero_division=0),
        "F1": f1_score(y_eval, predictions, zero_division=0),
        "Accuracy": accuracy_score(y_eval, predictions)
    }
    print("Confusion matrix (rows=actual, columns=predicted):")
    print(confusion_matrix(y_eval, predictions))
    print(classification_report(y_eval, predictions, zero_division=0))
    return metrics, probabilities, predictions

log_metrics, log_prob, log_pred = evaluate_model(logistic_model, X_test, y_test)
xgb_metrics, xgb_prob, xgb_pred = evaluate_model(xgb_model, X_test, y_test)

comparison = pd.DataFrame({
    "Logistic Regression": log_metrics,
    "XGBoost": xgb_metrics
}).T
display(comparison.round(3))

dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train, y_train)
dummy_pred = dummy.predict(X_test)
print("Majority-class baseline accuracy:", round(accuracy_score(y_test, dummy_pred), 3))
print("Majority-class baseline recall:", round(recall_score(y_test, dummy_pred, zero_division=0), 3))

precision, recall, thresholds = precision_recall_curve(y_test, xgb_prob)
plt.figure(figsize=(7, 5))
plt.plot(recall, precision)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("XGBoost Precision–Recall Curve")
plt.tight_layout()
plt.show()

## Step 11 — Model interpretation with SHAP

This optional step explains which features contribute to model predictions. It does not establish causality. SHAP can be memory-intensive for wide one-hot encoded data, so the example uses a sample of test observations.

In [ ]:
# If needed, install: %pip install shap
import shap

fitted_preprocessor = xgb_model.named_steps["preprocessor"]
X_test_transformed = fitted_preprocessor.transform(X_test)
feature_names_xgb = fitted_preprocessor.get_feature_names_out()

rng = np.random.default_rng(42)
sample_size = min(1000, X_test_transformed.shape[0])
sample_indices = rng.choice(X_test_transformed.shape[0], sample_size, replace=False)
X_shap = X_test_transformed[sample_indices]

fitted_xgb = xgb_model.named_steps["model"]
explainer = shap.TreeExplainer(fitted_xgb)
shap_values = explainer.shap_values(X_shap)

shap.summary_plot(
    shap_values, X_shap,
    feature_names=feature_names_xgb,
    plot_type="bar"
)
shap.summary_plot(
    shap_values, X_shap,
    feature_names=feature_names_xgb
)

## Step 12 — Business application and limitations

Use the model to prioritize bookings for further investigation or intervention testing. The risk thresholds below are illustrative, not validated business rules.

In [ ]:
risk_df = X_test.copy()
risk_df["actual_customer_cancelled"] = y_test.to_numpy()
risk_df["predicted_cancel_probability"] = xgb_prob

risk_df["risk_category"] = pd.cut(
    risk_df["predicted_cancel_probability"],
    bins=[-np.inf, 0.10, 0.25, 0.50, np.inf],
    labels=["Low", "Moderate", "High", "Very High"]
)

risk_summary = risk_df.groupby("risk_category", observed=True).agg(
    bookings=("actual_customer_cancelled", "size"),
    actual_cancellations=("actual_customer_cancelled", "sum"),
    average_predicted_risk=("predicted_cancel_probability", "mean"),
    actual_cancellation_rate=("actual_customer_cancelled", "mean")
)
display(risk_summary.round(3))

high_risk = risk_df[
    risk_df["risk_category"].isin(["High", "Very High"])
].copy()

print("Bookings in illustrative high/very-high risk groups:", len(high_risk))
print("Average predicted risk:",
      round(high_risk["predicted_cancel_probability"].mean(), 3)
      if len(high_risk) else "No bookings in these groups")

for col in ["Vehicle Type", "Pickup Location", "Payment Method", "hour"]:
    if col in high_risk.columns:
        print(f"\nTop values for {col} among high-risk bookings:")
        display(high_risk[col].value_counts().head(10).to_frame("bookings"))

## Suggested conclusions template

After running the notebook, fill in these statements using your actual results:

1. **Predictive performance:** XGBoost achieved ROC-AUC of ___ and average precision of ___, compared with ___ for Logistic Regression.
2. **Useful predictors:** The leading predictive features were ___, based on the fitted model and SHAP analysis.
3. **Operational implication:** The model could help prioritize bookings for tests of clearer ETA communication, alternative vehicle suggestions, or driver-matching changes.
4. **Limitations:** The dataset is observational; prediction does not establish causation. The primary analysis excludes other booking outcomes, and the timing of some variables needs verification.
5. **Next step:** Run a prospective randomized experiment to test whether a specific intervention reduces cancellation.

Do not claim an intervention prevents cancellations until it has been tested.